# GDELT global geographic graph

**Source-only archival notebook — exploratory research.** Exploratory code for parsing a GDELT graph index. Requires external source manifest and input files; code and results have not been independently rerun.

All saved outputs and execution counts were removed before public release. No source dataset, independent replication, deployable trading strategy, or verified investment performance is represented here.

Some original data-access paths and dependencies remain unresolved; code may require adaptation to run. Do not interpret results as causal or as a trading signal without additional validation.


In [ ]:
import re, requests, pandas as pd
with open("MASTERFILELIST.TXT", "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

rows = []
for line in lines:
    parts = line.strip().split()
    if len(parts) < 3: 
        continue
    url = next((p for p in parts if p.startswith("http")), None)
    md5 = next((p for p in parts if re.fullmatch(r"[0-9a-fA-F]{32}", p)), None)
    size = next((p for p in parts if p.isdigit()), None)
    if url and md5 and size:
        rows.append({"filename": parts[0], "bytes": int(size), "md5": md5, "url": url})

df = pd.DataFrame(rows)
df["GB"] = df["bytes"] / (1024**3)
df.head()

In [ ]:
print(len(rows))  # how many rows parsed?
print(df.columns)  # what columns exist?
print(df.head())   # preview the first rows

In [ ]:
from pathlib import Path

PATH = Path("MASTERFILELIST.TXT")  # adjust as needed

encodings_to_try = [
    "utf-8",
    "utf-8-sig",
    "latin-1",
    "cp1252",
    "iso-8859-1",
    "ascii"
]

for enc in encodings_to_try:
    try:
        text = PATH.read_text(encoding=enc)
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        print(f"Encoding: {enc} | Lines: {len(lines)}")
        print("First 3 lines:", lines[:3])
        print("-" * 50)
    except Exception as e:
        print(f"Encoding: {enc} FAILED — {e}")

In [ ]:
import re, pandas as pd
from pathlib import Path

PATH = Path("MASTERFILELIST.TXT")  # your file with 1,979 URLs
lines = [ln.strip() for ln in PATH.read_text(encoding="utf-8").splitlines() if ln.strip()]

# Build a DataFrame from the URL list
df = pd.DataFrame({"url": lines})
df["filename"] = df["url"].str.rsplit("/", n=1).str[-1]
# extract yyyymm (e.g., 201704) for dating
df["yyyymm"] = df["filename"].str.extract(r"(\d{6})").astype(int)
df["date"] = pd.to_datetime(df["yyyymm"].astype(str) + "01", format="%Y%m%d")

df.head()

In [ ]:
import requests

def content_length(u):
    try:
        r = requests.head(u, allow_redirects=True, timeout=15, verify=False)
        return int(r.headers.get("Content-Length")) if r.ok and r.headers.get("Content-Length") else None
    except Exception:
        return None

# sample a few to avoid hammering the server
sample_ix = df.sample(10, random_state=0).index
df.loc[sample_ix, "bytes"] = df.loc[sample_ix, "url"].map(content_length)
df["GB"] = df["bytes"] / (1024**3)
df.loc[sample_ix, ["filename","GB"]]

In [ ]:
import requests, gzip, io, pandas as pd

u = df.iloc[0]["20200927.ggg.v1.english.json.gz"]  # pick any row
bin_ = requests.get(u, stream=True, verify=False).content  # exploration only; set verify=True if your SSL works
with gzip.GzipFile(fileobj=io.BytesIO(bin_)) as gz:
    # GDELT GGG monthly dumps are JSON Lines (one JSON per line)
    g = pd.read_json(gz, lines=True)  # loads entire month; for memory safety, stream in chunks if needed
g.head()